Cell 1

In [7]:
import requests
import pandas as pd

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

OPENFDA_BASE = "https://api.fda.gov/drug/ndc.json"
DAILYMED_BASE = "https://dailymed.nlm.nih.gov/dailymed/services/v2"

#Cell 2

In [8]:
class OpenFDAClient:

    def search_by_brand(self, drug_name):

        try:

            response = requests.get(
                OPENFDA_BASE,
                params={
                    "search": f'brand_name:"{drug_name}"',
                    "limit": 100
                },
                timeout=30
            )

            response.raise_for_status()

            return response.json().get(
                "results",
                []
            )

        except Exception as e:

            print(
                f"Brand Search Error: {e}"
            )

            return []

    def search_by_generic(self, drug_name):

        try:

            response = requests.get(
                OPENFDA_BASE,
                params={
                    "search": f'generic_name:"{drug_name}"',
                    "limit": 100
                },
                timeout=30
            )

            response.raise_for_status()

            return response.json().get(
                "results",
                []
            )

        except Exception as e:

            print(
                f"Generic Search Error: {e}"
            )

            return []

    def search_by_ndc(self, ndc):

        searches = [

            f'package_ndc:"{ndc}"',

            f'product_ndc:"{ndc}"'
        ]

        for search_string in searches:

            try:

                response = requests.get(
                    OPENFDA_BASE,
                    params={
                        "search": search_string,
                        "limit": 10
                    },
                    timeout=30
                )

                response.raise_for_status()

                results = response.json().get(
                    "results",
                    []
                )

                if results:

                    return results

            except:
                pass

        return []

    def build_products_df(
        self,
        records
    ):

        rows = []

        for record in records:

            rows.append({

                "product_ndc":
                    record.get(
                        "product_ndc"
                    ),

                "brand_name":
                    record.get(
                        "brand_name"
                    ),

                "generic_name":
                    record.get(
                        "generic_name"
                    ),

                "manufacturer":
                    record.get(
                        "labeler_name"
                    ),

                "dosage_form":
                    record.get(
                        "dosage_form"
                    ),

                "route":
                    ", ".join(
                        record.get(
                            "route",
                            []
                        )
                    ),

                "marketing_category":
                    record.get(
                        "marketing_category"
                    ),

                "product_type":
                    record.get(
                        "product_type"
                    ),

                "application_number":
                    record.get(
                        "application_number"
                    ),

                "dea_schedule":
                    record.get(
                        "dea_schedule"
                    ),

                "start_marketing_date":
                    record.get(
                        "start_marketing_date"
                    ),

                "end_marketing_date":
                    record.get(
                        "end_marketing_date"
                    )
            })

        return (
            pd.DataFrame(rows)
            .drop_duplicates()
        )

    def build_packages_df(
        self,
        records
    ):

        rows = []

        for record in records:

            product_ndc = record.get(
                "product_ndc"
            )

            for package in record.get(
                "packaging",
                []
            ):

                rows.append({

                    "product_ndc":
                        product_ndc,

                    "package_ndc":
                        package.get(
                            "package_ndc"
                        ),

                    "package_description":
                        package.get(
                            "description"
                        ),

                    "sample":
                        package.get(
                            "sample"
                        )
                })

        return (
            pd.DataFrame(rows)
            .drop_duplicates()
        )

#Cell 3

In [9]:
class DailyMedClient:

    def get_ndc_record(
        self,
        ndc
    ):

        try:

            response = requests.get(
                f"{DAILYMED_BASE}/ndcs/{ndc}.json",
                timeout=30
            )

            if response.status_code != 200:

                return {}

            data = response.json().get(
                "data",
                []
            )

            if len(data) == 0:

                return {}

            return data[0]

        except:

            return {}

    def build_dailymed_df(
        self,
        ndc_list
    ):

        rows = []

        for ndc in ndc_list:

            record = self.get_ndc_record(
                ndc
            )

            if record:

                rows.append({

                    "package_ndc":
                        ndc,

                    "setid":
                        record.get(
                            "setid"
                        ),

                    "title":
                        record.get(
                            "title"
                        ),

                    "published_date":
                        record.get(
                            "published_date"
                        )
                })

        return pd.DataFrame(rows)

Cell 4

In [10]:
class DrugMaster:

    def __init__(self):

        self.fda = OpenFDAClient()

        self.dailymed = DailyMedClient()

    def search_drug(
        self,
        drug_name
    ):

        print(
            f"Searching for {drug_name}"
        )

        records = (
            self.fda.search_by_brand(
                drug_name
            )
        )

        if len(records) == 0:

            records = (
                self.fda.search_by_generic(
                    drug_name
                )
            )

        products_df = (
            self.fda.build_products_df(
                records
            )
        )

        packages_df = (
            self.fda.build_packages_df(
                records
            )
        )

        dailymed_df = pd.DataFrame()

        if not packages_df.empty:

            ndcs = (

                packages_df[
                    "package_ndc"
                ]

                .dropna()

                .unique()

                .tolist()
            )

            dailymed_df = (
                self.dailymed
                .build_dailymed_df(
                    ndcs
                )
            )


        return {

            "products":
                products_df,

            "packages":
                packages_df,

            "dailymed":
                dailymed_df,
                
            "_raw_fda_records":
                records
            
        }

    def search_ndc(
        self,
        ndc
    ):

        records = (
            self.fda.search_by_ndc(
                ndc
            )
        )

        products_df = (
            self.fda.build_products_df(
                records
            )
        )

        packages_df = (
            self.fda.build_packages_df(
                records
            )
        )

        dailymed_df = (
            self.dailymed
            .build_dailymed_df(
                [ndc]
            )
        )

        return {

            "products":
                products_df,

            "packages":
                packages_df,

            "dailymed":
                dailymed_df,
            
            "_raw_fda_records":
                records
        }

Cell 5

In [11]:
def show_results(results):

    for name, df in results.items():

        print("\n")
        print("=" * 80)
        print(name.upper())
        print("=" * 80)

        print(
            f"Rows: {len(df)}"
        )

        display(df)

Cell 6

In [12]:
def export_excel(
    results,
    filename
):

    with pd.ExcelWriter(
        filename,
        engine="openpyxl"
    ) as writer:

        for sheet, df in results.items():

            df.to_excel(
                writer,
                sheet_name=sheet[:31],
                index=False
            )

    print(
        f"Saved {filename}"
    )

Cell 7

In [13]:
drug_name = "Opdivo"

lookup = DrugMaster()

results = lookup.search_drug(
    drug_name
)

show_results(results)


Searching for Opdivo


PRODUCTS
Rows: 6


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
1,0003-3772,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None
3,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None
4,0003-3734,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
5,0003-3774,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None




PACKAGES
Rows: 6


,product_ndc,package_ndc,package_description,sample
0,0003-3756,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False
1,0003-3772,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",False
2,0003-6120,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False
3,0003-3120,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False
4,0003-3734,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",False
5,0003-3774,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",False




DAILYMED
Rows: 0


""


Cell 8 - Call this cell only if you want a specific NDC lookup

In [29]:
ndc_results = lookup.search_ndc(
    "0003-3756"
)

show_results(ndc_results)



PRODUCTS
Rows: 1


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None




PACKAGES
Rows: 1


,product_ndc,package_ndc,package_description,sample
0,0003-3756,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False




DAILYMED
Rows: 0


""


Cell 9

In [14]:
import os

def export_excel(
    results,
    drug_name
):

    filename = (
        f"DrugMaster_{drug_name}.xlsx"
        .replace(" ", "_")
        .replace("/", "_")
    )

    full_path = os.path.abspath(filename)

    with pd.ExcelWriter(
        filename,
        engine="openpyxl"
    ) as writer:

        for sheet, df in results.items():

            df.to_excel(
                writer,
                sheet_name=sheet[:31],
                index=False
            )

    print(f"Saved: {full_path}")

export_excel(
    results,
    drug_name
)

Saved: C:\Users\barry.peterson\OneDrive - Integra Connect LLC\Documents\Pharmacy Reports\Master Drug File\DrugMaster_Opdivo.xlsx


Cell 10

In [15]:
class RxNormClient:

    BASE_URL = "https://rxnav.nlm.nih.gov/REST"

    def get_rxcui(self, drug_name):

        try:

            response = requests.get(
                f"{self.BASE_URL}/rxcui.json",
                params={
                    "name": drug_name
                },
                timeout=30
            )

            response.raise_for_status()

            return (
                response.json()
                .get("idGroup", {})
                .get("rxnormId", [])
            )

        except Exception as e:

            print(
                f"RxNorm Error: {e}"
            )

            return []

    def get_properties(
        self,
        rxcui
    ):

        try:

            response = requests.get(
                f"{self.BASE_URL}/rxcui/{rxcui}/properties.json",
                timeout=30
            )

            response.raise_for_status()

            return (
                response.json()
                .get("properties", {})
            )

        except:

            return {}

    def get_related_concepts(
        self,
        rxcui
    ):

        ttys = [

            "IN",
            "MIN",
            "PIN",

            "SCD",
            "SBD",

            "SCDF",
            "SCDC",

            "GPCK",
            "BPCK"
        ]

        rows = []

        for tty in ttys:

            try:

                response = requests.get(
                    f"{self.BASE_URL}/rxcui/{rxcui}/related.json",
                    params={
                        "tty": tty
                    },
                    timeout=30
                )

                groups = (
                    response.json()
                    .get(
                        "relatedGroup",
                        {}
                    )
                    .get(
                        "conceptGroup",
                        []
                    )
                )

                for group in groups:

                    for concept in group.get(
                        "conceptProperties",
                        []
                    ):

                        rows.append({

                            "source_rxcui":
                                rxcui,

                            "related_rxcui":
                                concept.get(
                                    "rxcui"
                                ),

                            "name":
                                concept.get(
                                    "name"
                                ),

                            "tty":
                                tty
                        })

            except:
                pass

        return pd.DataFrame(rows)

Cell 11

In [16]:
def rxnorm_lookup(
    drug_name
):

    rx = RxNormClient()

    rxcuis = rx.get_rxcui(
        drug_name
    )

    summary_rows = []

    concept_rows = []

    for rxcui in rxcuis:

        props = rx.get_properties(
            rxcui
        )

        summary_rows.append({

            "rxcui":
                rxcui,

            "name":
                props.get("name"),

            "tty":
                props.get("tty"),

            "synonym":
                props.get("synonym")
        })

        concept_df = (
            rx.get_related_concepts(
                rxcui
            )
        )

        if not concept_df.empty:

            concept_rows.extend(
                concept_df.to_dict(
                    "records"
                )
            )

    return {

        "rxnorm_summary":
            pd.DataFrame(
                summary_rows
            ),

        "rxnorm_concepts":
            pd.DataFrame(
                concept_rows
            )
    }

Cell 12

In [17]:
rx_results = rxnorm_lookup(
    drug_name
)

for name, df in rx_results.items():

    print("\n")
    print(name)
    print("-" * 50)

    display(df.head(20))



rxnorm_summary
--------------------------------------------------


,rxcui,name,tty,synonym
0,1597881,Opdivo,BN,




rxnorm_concepts
--------------------------------------------------


,source_rxcui,related_rxcui,name,tty
0,1597881,1597876,nivolumab,IN
1,1597881,1657190,4 ML nivolumab 10 MG/ML Injection,SCD
2,1597881,1657195,10 ML nivolumab 10 MG/ML Injection,SCD
3,1597881,1991412,24 ML nivolumab 10 MG/ML Injection,SCD
4,1597881,2569080,12 ML nivolumab 10 MG/ML Injection,SCD
5,1597881,1657192,4 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
6,1597881,1657196,10 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
7,1597881,1991413,24 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
8,1597881,2569081,12 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
9,1597881,1657189,nivolumab Injection,SCDF


Cell 13

In [18]:
drug_results = lookup.search_drug(
    drug_name
)

rx_results = rxnorm_lookup(
    drug_name
)

combined_results = {

    **drug_results,

    **rx_results
}

Searching for Opdivo


Cell 14

In [19]:
show_results(
    combined_results
)



PRODUCTS
Rows: 6


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
1,0003-3772,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None
3,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None
4,0003-3734,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
5,0003-3774,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None




PACKAGES
Rows: 6


,product_ndc,package_ndc,package_description,sample
0,0003-3756,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False
1,0003-3772,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",False
2,0003-6120,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False
3,0003-3120,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False
4,0003-3734,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",False
5,0003-3774,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",False




DAILYMED
Rows: 0


""




RXNORM_SUMMARY
Rows: 1


,rxcui,name,tty,synonym
0,1597881,Opdivo,BN,




RXNORM_CONCEPTS
Rows: 11


,source_rxcui,related_rxcui,name,tty
0,1597881,1597876,nivolumab,IN
1,1597881,1657190,4 ML nivolumab 10 MG/ML Injection,SCD
2,1597881,1657195,10 ML nivolumab 10 MG/ML Injection,SCD
3,1597881,1991412,24 ML nivolumab 10 MG/ML Injection,SCD
4,1597881,2569080,12 ML nivolumab 10 MG/ML Injection,SCD
5,1597881,1657192,4 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
6,1597881,1657196,10 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
7,1597881,1991413,24 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
8,1597881,2569081,12 ML nivolumab 10 MG/ML Injection [Opdivo],SBD
9,1597881,1657189,nivolumab Injection,SCDF


Cell 15 - Ingredients

In [20]:
def build_ingredients_df(records):

    rows = []

    for record in records:

        product_ndc = record.get(
            "product_ndc"
        )

        brand_name = record.get(
            "brand_name"
        )

        generic_name = record.get(
            "generic_name"
        )

        for ingredient in record.get(
            "active_ingredients",
            []
        ):

            rows.append({

                "product_ndc":
                    product_ndc,

                "brand_name":
                    brand_name,

                "generic_name":
                    generic_name,

                "ingredient_name":
                    ingredient.get(
                        "name"
                    ),

                "strength":
                    ingredient.get(
                        "strength"
                    )
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Cell 16 - Build Ingredients

In [21]:
fda_records = lookup.fda.search_by_brand(
    drug_name
)

if not fda_records:

    fda_records = lookup.fda.search_by_generic(
        drug_name
    )

ingredients_df = build_ingredients_df(
    fda_records
)

display(
    ingredients_df
)

,product_ndc,brand_name,generic_name,ingredient_name,strength
0,0003-3756,OPDIVO,nivolumab,NIVOLUMAB,10 mg/mL
1,0003-3772,OPDIVO,nivolumab,NIVOLUMAB,10 mg/mL
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,HYALURONIDASE (HUMAN RECOMBINANT),2000 U/mL
3,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,NIVOLUMAB,120 mg/mL
4,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,HYALURONIDASE (HUMAN RECOMBINANT),2000 U/mL
5,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,NIVOLUMAB,120 mg/mL
6,0003-3734,OPDIVO,nivolumab,NIVOLUMAB,10 mg/mL
7,0003-3774,OPDIVO,nivolumab,NIVOLUMAB,10 mg/mL


Cell 17 - Pharmacologic Class

In [22]:
def build_pharm_class_df(records):

    rows = []

    for record in records:

        classes = record.get(
            "pharm_class",
            []
        )

        for pharm_class in classes:

            rows.append({

                "product_ndc":
                    record.get(
                        "product_ndc"
                    ),

                "brand_name":
                    record.get(
                        "brand_name"
                    ),

                "pharm_class":
                    pharm_class
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Cell 18 - Test Pharmacologic Class

In [23]:
pharm_class_df = build_pharm_class_df(
    fda_records
)

display(
    pharm_class_df
)

,product_ndc,brand_name,pharm_class
0,0003-3756,OPDIVO,Programmed Death Receptor-1 Blocking Antibody [EPC]
1,0003-3756,OPDIVO,Programmed Death Receptor-1-directed Antibody Interactions [MoA]
2,0003-3772,OPDIVO,Programmed Death Receptor-1 Blocking Antibody [EPC]
3,0003-3772,OPDIVO,Programmed Death Receptor-1-directed Antibody Interactions [MoA]
4,0003-6120,OPDIVO QVANTIG,Endoglycosidase [EPC]
5,0003-6120,OPDIVO QVANTIG,Glycoside Hydrolases [CS]
6,0003-6120,OPDIVO QVANTIG,Programmed Death Receptor-1 Blocking Antibody [EPC]
7,0003-6120,OPDIVO QVANTIG,Programmed Death Receptor-1-directed Antibody Interactions [MoA]
8,0003-3120,OPDIVO QVANTIG,Endoglycosidase [EPC]
9,0003-3120,OPDIVO QVANTIG,Glycoside Hydrolases [CS]


Cell 19 - Product Master Table

In [24]:
products_df = results["products"]
packages_df = results["packages"]

master_df = products_df.merge(

    packages_df,

    on="product_ndc",

    how="left"
)

if not ingredients_df.empty:

    master_df = master_df.merge(

        ingredients_df,

        on=[
            "product_ndc",
            "brand_name",
            "generic_name"
        ],

        how="left"
    )

display(
    master_df
)

,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date,package_ndc,package_description,sample,ingredient_name,strength
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False,NIVOLUMAB,10 mg/mL
1,0003-3772,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",False,NIVOLUMAB,10 mg/mL
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False,HYALURONIDASE (HUMAN RECOMBINANT),2000 U/mL
3,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False,NIVOLUMAB,120 mg/mL
4,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False,HYALURONIDASE (HUMAN RECOMBINANT),2000 U/mL
5,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False,NIVOLUMAB,120 mg/mL
6,0003-3734,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",False,NIVOLUMAB,10 mg/mL
7,0003-3774,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",False,NIVOLUMAB,10 mg/mL


Cell 20

In [25]:
def combine_results(
    result_list
):

    combined = {}

    for result in result_list:

        for key, df in result.items():

            if key not in combined:

                combined[key] = []

            combined[key].append(df)

    final = {}

    for key, dfs in combined.items():

        final[key] = (
            pd.concat(
                dfs,
                ignore_index=True
            )
            .drop_duplicates()
        )

    return final

Cell 21

In [26]:
def search_drugs(
    lookup,
    drug_list
):

    all_results = []

    total = len(drug_list)

    for i, drug in enumerate(
        drug_list,
        start=1
    ):

        print(
            f"[{i}/{total}] "
            f"Searching {drug}"
        )

        result = lookup.search_drug(
            drug
        )

        all_results.append(
            result
        )

    return combine_results(
        all_results
    )

Cell 22

In [27]:
lookup = DrugMaster()

In [28]:
drug_list = [

    "Opdivo",
    "Keytruda",
    "Yervoy"
]

multi_results = search_drugs(
    lookup,
    drug_list
)

show_results(
    multi_results
)

[1/3] Searching Opdivo
Searching for Opdivo
[2/3] Searching Keytruda
Searching for Keytruda
[3/3] Searching Yervoy
Searching for Yervoy


PRODUCTS
Rows: 11


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
1,0003-3772,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None
3,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None
4,0003-3734,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
5,0003-3774,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None
6,0006-5083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None
7,0006-3026,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125514,None,None,None
8,0006-3083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None
9,0003-2328,YERVOY,ipilimumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125377,None,None,None




PACKAGES
Rows: 12


,product_ndc,package_ndc,package_description,sample
0,0003-3756,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False
1,0003-3772,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",False
2,0003-6120,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False
3,0003-3120,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False
4,0003-3734,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",False
5,0003-3774,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",False
6,0006-5083,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),False
7,0006-3026,0006-3026-02,1 VIAL in 1 CARTON (0006-3026-02) / 4 mL in 1 VIAL (0006-3026-01),False
8,0006-3026,0006-3026-04,2 VIAL in 1 CARTON (0006-3026-04) / 4 mL in 1 VIAL (0006-3026-01),False
9,0006-3083,0006-3083-01,1 VIAL in 1 CARTON (0006-3083-01) / 2.4 mL in 1 VIAL (0006-3083-99),False




DAILYMED
Rows: 0


""


In [6]:
print("DrugMaster" in globals())
print("lookup" in globals())

False
False


Cell 24 - Build Portfolio Master

In [29]:
def build_portfolio_master(results):

    products_df = results["products"].copy()

    packages_df = results["packages"].copy()

    master_df = products_df

    if not packages_df.empty:

        master_df = master_df.merge(

            packages_df,

            on="product_ndc",

            how="left"
        )

    return (
        master_df
        .drop_duplicates()
        .reset_index(drop=True)
    )

Cell 25 - Build Portfolio

In [30]:
portfolio_master_df = build_portfolio_master(
    multi_results
)

print(
    f"Rows: {len(portfolio_master_df)}"
)

display(
    portfolio_master_df.head(25)
)

Rows: 12


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date,package_ndc,package_description,sample
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False
1,0003-3772,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",False
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False
3,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False
4,0003-3734,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",False
5,0003-3774,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",False
6,0006-5083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),False
7,0006-3026,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125514,None,None,None,0006-3026-02,1 VIAL in 1 CARTON (0006-3026-02) / 4 mL in 1 VIAL (0006-3026-01),False
8,0006-3026,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125514,None,None,None,0006-3026-04,2 VIAL in 1 CARTON (0006-3026-04) / 4 mL in 1 VIAL (0006-3026-01),False
9,0006-3083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None,0006-3083-01,1 VIAL in 1 CARTON (0006-3083-01) / 2.4 mL in 1 VIAL (0006-3083-99),False


Cell 26 - Data Quality Check

In [31]:
print(
    f"Unique Products: "
    f"{portfolio_master_df['product_ndc'].nunique()}"
)

print(
    f"Unique Packages: "
    f"{portfolio_master_df['package_ndc'].nunique()}"
)

print(
    f"Unique Brands: "
    f"{portfolio_master_df['brand_name'].nunique()}"
)

print(
    f"Unique Manufacturers: "
    f"{portfolio_master_df['manufacturer'].nunique()}"
)

Unique Products: 11
Unique Packages: 12
Unique Brands: 5
Unique Manufacturers: 2


Cell 27 - Search Example

In [32]:
portfolio_master_df[
    [
        "brand_name",
        "generic_name",
        "manufacturer",
        "package_ndc",
        "package_description"
    ]
].head(20)

,brand_name,generic_name,manufacturer,package_ndc,package_description
0,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE"
1,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE"
2,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE"
3,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE"
4,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE"
5,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE"
6,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99)
7,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,0006-3026-02,1 VIAL in 1 CARTON (0006-3026-02) / 4 mL in 1 VIAL (0006-3026-01)
8,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,0006-3026-04,2 VIAL in 1 CARTON (0006-3026-04) / 4 mL in 1 VIAL (0006-3026-01)
9,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,0006-3083-01,1 VIAL in 1 CARTON (0006-3083-01) / 2.4 mL in 1 VIAL (0006-3083-99)


Cell 28 - Export Portfolio

In [33]:
export_excel(
    multi_results,
    "ImmunoOncologyPortfolio"
)

Saved: C:\Users\barry.peterson\OneDrive - Integra Connect LLC\Documents\Pharmacy Reports\Master Drug File\DrugMaster_ImmunoOncologyPortfolio.xlsx


Cell 29 - Add Ingredient Info

In [34]:
def build_ingredients_df(records):

    rows = []

    for record in records:

        for ingredient in record.get(
            "active_ingredients",
            []
        ):

            rows.append({

                "product_ndc":
                    record.get(
                        "product_ndc"
                    ),

                "ingredient_name":
                    ingredient.get(
                        "name"
                    ),

                "strength":
                    ingredient.get(
                        "strength"
                    )
            })

    return (
        pd.DataFrame(rows)
        .drop_duplicates()
    )

Cell 30 - Multi-source Data

In [35]:
opdivo_records = (
    lookup.fda.search_by_brand(
        "Opdivo"
    )
)

ingredients_df = build_ingredients_df(
    opdivo_records
)

display(
    ingredients_df
)

,product_ndc,ingredient_name,strength
0,0003-3756,NIVOLUMAB,10 mg/mL
1,0003-3772,NIVOLUMAB,10 mg/mL
2,0003-6120,HYALURONIDASE (HUMAN RECOMBINANT),2000 U/mL
3,0003-6120,NIVOLUMAB,120 mg/mL
4,0003-3120,HYALURONIDASE (HUMAN RECOMBINANT),2000 U/mL
5,0003-3120,NIVOLUMAB,120 mg/mL
6,0003-3734,NIVOLUMAB,10 mg/mL
7,0003-3774,NIVOLUMAB,10 mg/mL


Ingredients Table Builder

In [37]:
def build_ingredients_df(results):

    products_df = results["products"]

    rows = []

    for _, row in products_df.iterrows():

        product_ndc = row.get(
            "product_ndc"
        )

        brand_name = row.get(
            "brand_name"
        )

        generic_name = row.get(
            "generic_name"
        )

        rows.append({

            "product_ndc":
                product_ndc,

            "brand_name":
                brand_name,

            "generic_name":
                generic_name,

            "ingredient_name":
                generic_name
        })

    ingredients_df = pd.DataFrame(
        rows
    )

    return (
        ingredients_df
        .drop_duplicates()
        .reset_index(drop=True)
    )

Build Ingredients Table

In [38]:
ingredients_df = build_ingredients_df(
    multi_results
)

print(
    f"Ingredient Rows: {len(ingredients_df)}"
)

display(
    ingredients_df.head(25)
)

Ingredient Rows: 11


,product_ndc,brand_name,generic_name,ingredient_name
0,0003-3756,OPDIVO,nivolumab,nivolumab
1,0003-3772,OPDIVO,nivolumab,nivolumab
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy
3,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy
4,0003-3734,OPDIVO,nivolumab,nivolumab
5,0003-3774,OPDIVO,nivolumab,nivolumab
6,0006-5083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,pembrolizumab and berahyaluronidase alfa-pmph
7,0006-3026,KEYTRUDA,pembrolizumab,pembrolizumab
8,0006-3083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,pembrolizumab and berahyaluronidase alfa-pmph
9,0003-2328,YERVOY,ipilimumab,ipilimumab


Drug Master Preview

In [39]:
portfolio_master_v2 = (
    portfolio_master_df
    .merge(
        ingredients_df,
        on=[
            "product_ndc",
            "brand_name",
            "generic_name"
        ],
        how="left"
    )
)

print(
    f"Rows: {len(portfolio_master_v2)}"
)

display(
    portfolio_master_v2.head(25)
)

Rows: 12


,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date,package_ndc,package_description,sample,ingredient_name
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab
1,0003-3772,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab and hyaluronidase-nvhy
3,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab and hyaluronidase-nvhy
4,0003-3734,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab
5,0003-3774,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab
6,0006-5083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),False,pembrolizumab and berahyaluronidase alfa-pmph
7,0006-3026,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125514,None,None,None,0006-3026-02,1 VIAL in 1 CARTON (0006-3026-02) / 4 mL in 1 VIAL (0006-3026-01),False,pembrolizumab
8,0006-3026,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125514,None,None,None,0006-3026-04,2 VIAL in 1 CARTON (0006-3026-04) / 4 mL in 1 VIAL (0006-3026-01),False,pembrolizumab
9,0006-3083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None,0006-3083-01,1 VIAL in 1 CARTON (0006-3083-01) / 2.4 mL in 1 VIAL (0006-3083-99),False,pembrolizumab and berahyaluronidase alfa-pmph


Package Description Parser

In [40]:
import re

def parse_package_description(description):

    result = {

        "package_quantity": None,

        "package_unit": None,

        "container_quantity": None,

        "container_type": None
    }

    if pd.isna(description):

        return result

    text = str(description).upper()

    matches = re.findall(
        r'(\d+)\s+([A-Z]+)',
        text
    )

    if len(matches) >= 1:

        result["package_quantity"] = int(
            matches[0][0]
        )

        result["package_unit"] = (
            matches[0][1]
        )

    if len(matches) >= 2:

        result["container_quantity"] = int(
            matches[1][0]
        )

        result["container_type"] = (
            matches[1][1]
        )

    return result

Build Package Analytics Table

In [41]:
package_analytics_df = (
    portfolio_master_v2.copy()
)

parsed = (
    package_analytics_df[
        "package_description"
    ]
    .fillna("")
    .apply(
        parse_package_description
    )
)

parsed_df = pd.DataFrame(
    parsed.tolist()
)

package_analytics_df = pd.concat(
    [
        package_analytics_df,
        parsed_df
    ],
    axis=1
)

display(
    package_analytics_df.head(25)
)

,product_ndc,brand_name,generic_name,manufacturer,dosage_form,route,marketing_category,product_type,application_number,dea_schedule,start_marketing_date,end_marketing_date,package_ndc,package_description,sample,ingredient_name,package_quantity,package_unit,container_quantity,container_type
0,0003-3756,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab,1,VIAL,1,CARTON
1,0003-3772,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab,1,VIAL,1,CARTON
2,0003-6120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab and hyaluronidase-nvhy,1,VIAL,1,CARTON
3,0003-3120,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.","INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761381,None,None,None,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab and hyaluronidase-nvhy,1,VIAL,1,CARTON
4,0003-3734,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab,1,VIAL,1,CARTON
5,0003-3774,OPDIVO,nivolumab,"E.R. Squibb & Sons, L.L.C.",INJECTION,INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125554,None,None,None,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",False,nivolumab,1,VIAL,1,CARTON
6,0006-5083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),False,pembrolizumab and berahyaluronidase alfa-pmph,1,VIAL,1,CARTON
7,0006-3026,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125514,None,None,None,0006-3026-02,1 VIAL in 1 CARTON (0006-3026-02) / 4 mL in 1 VIAL (0006-3026-01),False,pembrolizumab,1,VIAL,1,CARTON
8,0006-3026,KEYTRUDA,pembrolizumab,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",INTRAVENOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA125514,None,None,None,0006-3026-04,2 VIAL in 1 CARTON (0006-3026-04) / 4 mL in 1 VIAL (0006-3026-01),False,pembrolizumab,2,VIAL,1,CARTON
9,0006-3083,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,HUMAN PRESCRIPTION DRUG,BLA761467,None,None,None,0006-3083-01,1 VIAL in 1 CARTON (0006-3083-01) / 2.4 mL in 1 VIAL (0006-3083-99),False,pembrolizumab and berahyaluronidase alfa-pmph,1,VIAL,1,CARTON


In [42]:
package_analytics_df[
    [
        "brand_name",
        "package_description",
        "package_quantity",
        "package_unit",
        "container_quantity",
        "container_type"
    ]
].head(25)

,brand_name,package_description,package_quantity,package_unit,container_quantity,container_type
0,OPDIVO,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON
1,OPDIVO,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON
2,OPDIVO QVANTIG,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON
3,OPDIVO QVANTIG,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON
4,OPDIVO,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON
5,OPDIVO,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON
6,KEYTRUDA QLEX,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),1,VIAL,1,CARTON
7,KEYTRUDA,1 VIAL in 1 CARTON (0006-3026-02) / 4 mL in 1 VIAL (0006-3026-01),1,VIAL,1,CARTON
8,KEYTRUDA,2 VIAL in 1 CARTON (0006-3026-04) / 4 mL in 1 VIAL (0006-3026-01),2,VIAL,1,CARTON
9,KEYTRUDA QLEX,1 VIAL in 1 CARTON (0006-3083-01) / 2.4 mL in 1 VIAL (0006-3083-99),1,VIAL,1,CARTON


Build Drug Repository Table

In [43]:
def build_drug_repository(df):

    repository_df = df.copy()

    columns = [

        "brand_name",

        "generic_name",

        "ingredient_name",

        "manufacturer",

        "product_ndc",

        "package_ndc",

        "package_description",

        "package_quantity",

        "package_unit",

        "container_quantity",

        "container_type",

        "dosage_form",

        "route",

        "marketing_category",

        "application_number",

        "product_type"
    ]

    existing_columns = [

        col
        for col in columns
        if col in repository_df.columns
    ]

    repository_df = (
        repository_df[existing_columns]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    return repository_df

Create Repository

In [44]:
drug_repository_df = build_drug_repository(
    package_analytics_df
)

print(
    f"Repository Rows: {len(drug_repository_df)}"
)

print(
    f"Repository Columns: {len(drug_repository_df.columns)}"
)

display(
    drug_repository_df.head(25)
)

Repository Rows: 12
Repository Columns: 16


,brand_name,generic_name,ingredient_name,manufacturer,product_ndc,package_ndc,package_description,package_quantity,package_unit,container_quantity,container_type,dosage_form,route,marketing_category,application_number,product_type
0,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3756,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
1,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3772,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
2,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-6120,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761381,HUMAN PRESCRIPTION DRUG
3,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-3120,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761381,HUMAN PRESCRIPTION DRUG
4,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3734,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
5,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3774,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
6,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,0006-5083,0006-5083-01,1 VIAL in 1 CARTON (0006-5083-01) / 4.8 mL in 1 VIAL (0006-5083-99),1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761467,HUMAN PRESCRIPTION DRUG
7,KEYTRUDA,pembrolizumab,pembrolizumab,Merck Sharp & Dohme LLC,0006-3026,0006-3026-02,1 VIAL in 1 CARTON (0006-3026-02) / 4 mL in 1 VIAL (0006-3026-01),1,VIAL,1,CARTON,"INJECTION, SOLUTION",INTRAVENOUS,BLA,BLA125514,HUMAN PRESCRIPTION DRUG
8,KEYTRUDA,pembrolizumab,pembrolizumab,Merck Sharp & Dohme LLC,0006-3026,0006-3026-04,2 VIAL in 1 CARTON (0006-3026-04) / 4 mL in 1 VIAL (0006-3026-01),2,VIAL,1,CARTON,"INJECTION, SOLUTION",INTRAVENOUS,BLA,BLA125514,HUMAN PRESCRIPTION DRUG
9,KEYTRUDA QLEX,pembrolizumab and berahyaluronidase alfa-pmph,pembrolizumab and berahyaluronidase alfa-pmph,Merck Sharp & Dohme LLC,0006-3083,0006-3083-01,1 VIAL in 1 CARTON (0006-3083-01) / 2.4 mL in 1 VIAL (0006-3083-99),1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761467,HUMAN PRESCRIPTION DRUG


Repository Profiling

In [45]:
print(
    "Unique Brands:",
    drug_repository_df[
        "brand_name"
    ].nunique()
)

print(
    "Unique Generic Names:",
    drug_repository_df[
        "generic_name"
    ].nunique()
)

print(
    "Unique Manufacturers:",
    drug_repository_df[
        "manufacturer"
    ].nunique()
)

print(
    "Unique Product NDCs:",
    drug_repository_df[
        "product_ndc"
    ].nunique()
)

print(
    "Unique Package NDCs:",
    drug_repository_df[
        "package_ndc"
    ].nunique()
)

Unique Brands: 5
Unique Generic Names: 5
Unique Manufacturers: 2
Unique Product NDCs: 11
Unique Package NDCs: 12


Search Function Against Repository

In [46]:
def search_repository_by_brand(
    repository_df,
    brand_name
):

    return repository_df[
        repository_df[
            "brand_name"
        ]
        .str.contains(
            brand_name,
            case=False,
            na=False
        )
    ]

In [47]:
search_repository_by_brand(
    drug_repository_df,
    "Opdivo"
)

,brand_name,generic_name,ingredient_name,manufacturer,product_ndc,package_ndc,package_description,package_quantity,package_unit,container_quantity,container_type,dosage_form,route,marketing_category,application_number,product_type
0,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3756,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
1,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3772,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
2,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-6120,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761381,HUMAN PRESCRIPTION DRUG
3,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-3120,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761381,HUMAN PRESCRIPTION DRUG
4,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3734,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
5,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3774,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG


In [48]:
def search_repository_by_ingredient(
    repository_df,
    ingredient_name
):

    return repository_df[
        repository_df[
            "ingredient_name"
        ]
        .str.contains(
            ingredient_name,
            case=False,
            na=False
        )
    ]

In [49]:
search_repository_by_ingredient(
    drug_repository_df,
    "nivolumab"
)

,brand_name,generic_name,ingredient_name,manufacturer,product_ndc,package_ndc,package_description,package_quantity,package_unit,container_quantity,container_type,dosage_form,route,marketing_category,application_number,product_type
0,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3756,0003-3756-14,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3756-14) / 12 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
1,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3772,0003-3772-11,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3772-11) / 4 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
2,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-6120,0003-6120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-6120-01) / 5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761381,HUMAN PRESCRIPTION DRUG
3,OPDIVO QVANTIG,nivolumab and hyaluronidase-nvhy,nivolumab and hyaluronidase-nvhy,"E.R. Squibb & Sons, L.L.C.",0003-3120,0003-3120-01,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3120-01) / 2.5 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,"INJECTION, SOLUTION",SUBCUTANEOUS,BLA,BLA761381,HUMAN PRESCRIPTION DRUG
4,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3734,0003-3734-13,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3734-13) / 24 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG
5,OPDIVO,nivolumab,nivolumab,"E.R. Squibb & Sons, L.L.C.",0003-3774,0003-3774-12,"1 VIAL, SINGLE-DOSE in 1 CARTON (0003-3774-12) / 10 mL in 1 VIAL, SINGLE-DOSE",1,VIAL,1,CARTON,INJECTION,INTRAVENOUS,BLA,BLA125554,HUMAN PRESCRIPTION DRUG


In [50]:
drug_repository_df.to_csv(
    "DrugRepository.csv",
    index=False
)

print(
    "DrugRepository.csv saved"
)

DrugRepository.csv saved
